## A

State the input, output and clinical question in your own words:

The input of this task is greyscale images, FINISH THIS

Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from pathlib import Path
import sys
import importlib.util
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED_VALUE = 100
np.random.default_rng(RANDOM_SEED_VALUE)


Generator(PCG64) at 0x33FFA3840

In [32]:
dicom_files = list(Path("dataset").rglob("*.dcm"))
image_list = [pydicom.dcmread(file, stop_before_pixels=True) for file in dicom_files] #Don't get pixels

In [59]:
#Get the labelled CSV file as a Pandas dataframe
df = pd.read_csv("assignment1supportingfiles/assignment1_labels.csv")

mapping = pd.read_csv("assignment1supportingfiles/rsna_to_nih_mapping.csv")

print(f"Labels CSV file: Number of rows: {len(df)}")
#Drop the irrelevant columns
labels_df = df[["patientId", "Target", "StudyInstanceUID"]]


print(f"Number of duplicate patient Ids: {labels_df['patientId'].duplicated().sum()}")

labels_df = labels_df.drop_duplicates(subset="patientId")
print(f"Number of unique examinations and source patients: {len(labels_df)}")

print(f"Class distributions by count: {labels_df["Target"].value_counts()}")

print(f"Class distributions by percentage: {labels_df["Target"].value_counts(normalize=True) * 100}")



print(image_list[0])

# Set train test split sizes
train_percentage = 0.8
test_percentage = validation_percentage = (1 - train_percentage) / 2



labels_df = pd.merge(labels_df,mapping, how="inner", on="patientId", validate="1:1" )
labels_df = labels_df[["nih_patient_id", "Target", "nih_image_id"]]
print(labels_df.head())



Labels CSV file: Number of rows: 28989
Number of duplicate patient Ids: 3305
Number of unique examinations and source patients: 25684
Class distributions by count: Target
0    20025
1     5659
Name: count, dtype: int64
Class distributions by percentage: Target
0    77.966828
1    22.033172
Name: proportion, dtype: float64
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 202
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Secondary Capture Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.276.0.7230010.3.1.4.8323329.13028.1517874368.999134
(0002,0010) Transfer Syntax UID                 UI: JPEG Baseline (Process 1)
(0002,0012) Implementation Class UID            UI: 1.2.276.0.7230010.3.0.3.6.0
(0002,0013) Implementation Version Name         SH: 'OFFIS_DCMTK_360'
-------------------------------------------------
(0008,0005) Specific Character Set  

## B

In [76]:
sgkf = StratifiedGroupKFold(n_splits=round(1 / test_percentage), random_state=RANDOM_SEED_VALUE, shuffle=True)
train_id, test_id = next(sgkf.split(labels_df, labels_df["Target"], groups=labels_df["nih_patient_id"]))

train_val_df = labels_df.iloc[train_id]
test_df = labels_df.iloc[test_id]

#Do it again to get validation


sgkf = StratifiedGroupKFold(n_splits=round((1 - test_percentage) / validation_percentage), random_state=RANDOM_SEED_VALUE, shuffle=True)
train_id, val_id = next(sgkf.split(train_val_df, train_val_df["Target"], groups=train_val_df["nih_patient_id"]))
train_df = train_val_df.iloc[train_id]
val_df = train_val_df.iloc[val_id]

print(f"Count size of splits.\n train: {len(train_df)}\n validation: {len(val_df)}\n test: {len(test_df)} \n total: {len(val_df) + len(train_df) + len(test_df)}")
print(f"Percentage size of splits.\n train: {len(train_df) / len(labels_df)}\n validation: {len(val_df) / len(labels_df)}\n test: {len(test_df) / len(labels_df)}")
print("#" * 40)

print(f"Class distributions per split.\n train:{train_df["Target"].value_counts(normalize=True)}\n validation:{val_df["Target"].value_counts(normalize=True)}\n test:{test_df["Target"].value_counts(normalize=True)}")

set_train_df = set(train_df["nih_patient_id"])
set_val_df = set(val_df["nih_patient_id"])
set_test_df = set(test_df["nih_patient_id"])
assert set_train_df.isdisjoint(set_val_df)
assert set_train_df.isdisjoint(set_test_df)
assert set_val_df.isdisjoint(set_test_df)

Count size of splits.
 train: 20327
 validation: 2664
 test: 2693 
 total: 25684
Percentage size of splits.
 train: 0.7914265690702382
 validation: 0.10372216165706276
 test: 0.10485126927269896
########################################
Class distributions per split.
 train:Target
0    0.777242
1    0.222758
Name: proportion, dtype: float64
 validation:Target
0    0.792042
1    0.207958
Name: proportion, dtype: float64
 test:Target
0    0.785741
1    0.214259
Name: proportion, dtype: float64
